# Phase 2 — **Scheduled QAT** (Main Contribution)

**Hypothesis.** Standard QAT applies full quantization noise from step 1, forcing the model to handle the largest precision shock immediately. Gradually reducing the simulated bit-width over training — first FP32 → 16 → 8 → 4 — gives the weights time to adapt at each level before the next shock. We expect **lower KL divergence and lower perplexity at the same target bit-width** versus standard QAT.

We compare three schedules:

| Schedule | Curve | Intuition |
|---|---|---|
| **linear** | constant slope | predictable, no preference for any precision level |
| **cosine** | slow → fast → slow | spends extra time at endpoints (gentle entry to each new level) |
| **step** | hard drops with plateaus | maximum control over when each precision level kicks in |

**Inputs:** `/kaggle/input/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs:**
- `models/checkpoints/scheduled_qat_{linear,cosine,step}_int4/final.pt`
- `results/scheduled_qat_{linear,cosine,step}_int4/training_results.json`
- (Optional) Best schedule re-run at INT8 if `RUN_BEST_INT8=True`.

**Estimated time on T4:** 3 schedules × ~1.7 h ≈ **5 h** for INT4. Add ~1.7 h if INT8 is enabled.

## 1. Setup

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

WORKING_DIR  = "/kaggle/working"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
REPO_DATASET = "/kaggle/input/sqat-repo"
BASELINE_DIR = "/kaggle/input/sqat-baseline/results/baseline"

if not os.path.exists(REPO_DIR):
    if os.path.exists(REPO_DATASET):
        shutil.copytree(REPO_DATASET, REPO_DIR)
    else:
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), "FP32 logits not found — add sqat-baseline as input"
print(f"FP32 logits: {FP32_LOGITS}")

In [ ]:
!pip install -q -e ".[viz]"
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")

## 2. Schedule visualization

Before training, plot what each schedule actually does to the bit-width over time. The transitions are read from the YAML configs and converted to discrete fake-quant levels via `snap_bits()` — that mapping is what the model actually experiences.

Reading this plot:
- The **continuous** lines show the schedule's float bit-width.
- The **stepped** lines show the actual fake-quant level applied (32→None, 16→None, 8→INT8, 4→INT4).

In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt

from src.quantization.scheduler import build_scheduler, snap_bits
from src.utils.config_loader import load_config

def plot_schedules(name_to_cfg: dict, suptitle: str):
    fig, axes = plt.subplots(1, len(name_to_cfg), figsize=(5 * len(name_to_cfg), 4), sharey=True)
    if len(name_to_cfg) == 1:
        axes = [axes]
    for ax, (name, cfg_path) in zip(axes, name_to_cfg.items()):
        cfg = load_config(cfg_path)
        sched = build_scheduler(cfg.schedule, total_epochs=float(cfg.training.epochs))
        epochs = np.linspace(0, cfg.training.epochs, 400)
        cont = [sched.get_state(e).continuous_bits for e in epochs]
        snap = [snap_bits(c) or 32 for c in cont]
        ax.plot(epochs, cont, label="continuous", color="tab:blue")
        ax.step(epochs, snap, where="post", label="fake-quant level", color="tab:red", alpha=0.6)
        ax.set_title(f"{name} ({cfg.training.epochs} epoch{'s' if cfg.training.epochs != 1 else ''})")
        ax.set_xlabel("epoch"); ax.set_yticks([4, 8, 16, 32])
        ax.grid(True, alpha=0.3); ax.legend(loc="upper right")
    axes[0].set_ylabel("bit-width")
    fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

ORIGINAL_PATHS = {
    "linear": "configs/scheduled_qat/scheduled_linear_int4.yaml",
    "cosine": "configs/scheduled_qat/scheduled_cosine_int4.yaml",
    "step":   "configs/scheduled_qat/scheduled_step_int4.yaml",
}
plot_schedules(ORIGINAL_PATHS, "Original 3-epoch schedules (start_bits=32 → target_bits=4)")

## 3. Kaggle config overrides

Same overrides as notebook 03 (Standard QAT) so the comparison is apples-to-apples. The `schedule` block is preserved verbatim from the original config — it's the only thing that differs across the three runs.

In [ ]:
import yaml

MODEL_CACHE = f"{WORKING_DIR}/models/base"
KAGGLE_CFG  = Path(REPO_DIR) / "configs_kaggle" / "scheduled_qat"
KAGGLE_CFG.mkdir(parents=True, exist_ok=True)

# Toggle to also run the winning schedule at INT8 after the INT4 sweep.
RUN_BEST_INT8 = False

# Original configs target 3 epochs; we run 1 epoch on Kaggle. Compress every
# epoch-denominated field in the schedule by the same factor so the relative
# shape (warmup / active / stabilization) is preserved.
KAGGLE_EPOCHS    = 1
ORIGINAL_EPOCHS  = 3
SCHEDULE_SCALE   = KAGGLE_EPOCHS / ORIGINAL_EPOCHS

def patch_sqat_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = KAGGLE_EPOCHS
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 100
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{WORKING_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{WORKING_DIR}/models/gguf/"

    sch = cfg["schedule"]
    sch["warmup_epochs"]        = round(sch["warmup_epochs"]        * SCHEDULE_SCALE, 4)
    sch["stabilization_epochs"] = round(sch["stabilization_epochs"] * SCHEDULE_SCALE, 4)
    for t in sch.get("transitions") or []:
        t["epoch"] = round(t["epoch"] * SCHEDULE_SCALE, 4)

    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

kaggle_cfgs = {}
for name in ("linear", "cosine", "step"):
    src = f"configs/scheduled_qat/scheduled_{name}_int4.yaml"
    dst = KAGGLE_CFG / f"scheduled_{name}_int4.yaml"
    kaggle_cfgs[name] = patch_sqat_config(src, dst)
    print(f"  {name:7s} -> {dst}")

print("\nExample patched config (cosine):")
!cat {kaggle_cfgs['cosine']}

### Patched schedules — what the Kaggle runs actually use

The plot below shows the schedules after compression to one epoch. Compare to the 3-epoch originals above — same shape, smaller time axis. The model spends proportionally less time at each precision level than in the published configs.

In [ ]:
plot_schedules({k: str(v) for k, v in kaggle_cfgs.items()},
               f"Patched {KAGGLE_EPOCHS}-epoch schedules (Kaggle runs)")

## 4. Linear schedule — INT4

Constant rate of precision reduction. Simplest baseline schedule.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_linear = run_experiment(
    config_path=str(kaggle_cfgs["linear"]),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nLinear schedule INT4:")
for k, v in results_linear.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Cosine schedule — INT4

Slow start, fast middle, slow end. Should beat linear if the cosine LR analogy carries over to precision.

In [ ]:
results_cosine = run_experiment(
    config_path=str(kaggle_cfgs["cosine"]),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nCosine schedule INT4:")
for k, v in results_cosine.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 6. Step schedule — INT4

Hard drops at the configured transition epochs, with flat plateaus between. Each precision level gets a stable training period before the next drop.

In [ ]:
results_step = run_experiment(
    config_path=str(kaggle_cfgs["step"]),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nStep schedule INT4:")
for k, v in results_step.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 7. Schedule comparison (INT4)

The headline table for the thesis. Lower PPL and lower KLD = better. The winning schedule is fed into the optional INT8 cell below.

In [ ]:
import json
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"schedule": "linear", **{k: results_linear.get(k) for k in ("perplexity", "kl_divergence", "final_loss", "training_time_seconds")}},
    {"schedule": "cosine", **{k: results_cosine.get(k) for k in ("perplexity", "kl_divergence", "final_loss", "training_time_seconds")}},
    {"schedule": "step",   **{k: results_step.get(k)   for k in ("perplexity", "kl_divergence", "final_loss", "training_time_seconds")}},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / fp32["perplexity"]) - 1.0) * 100
df = df.round(4)
df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(df["schedule"], df["perplexity"], color=["tab:blue", "tab:orange", "tab:green"])
axes[0].axhline(fp32["perplexity"], linestyle="--", color="black", label=f"FP32 ({fp32['perplexity']:.2f})")
axes[0].set_ylabel("Perplexity"); axes[0].set_title("INT4 perplexity by schedule")
axes[0].legend()

axes[1].bar(df["schedule"], df["kl_divergence"], color=["tab:blue", "tab:orange", "tab:green"])
axes[1].set_ylabel("KL divergence vs FP32"); axes[1].set_title("INT4 KL divergence by schedule")

for ax in axes:
    ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

best = df.loc[df["kl_divergence"].idxmin(), "schedule"]
print(f"Best schedule (lowest KLD): {best}")

## 8. (Optional) Best schedule → INT8

If you have time left in the session, re-run the winning schedule at INT8 for the cross-method comparison in notebook 07. Set `RUN_BEST_INT8=True` in the override cell above.

In [ ]:
results_int8 = None
if RUN_BEST_INT8:
    src = f"configs/scheduled_qat/scheduled_{best}_int8.yaml"
    dst = KAGGLE_CFG / f"scheduled_{best}_int8.yaml"
    patch_sqat_config(src, dst)
    print(f"Running best schedule={best} at INT8...\n")
    results_int8 = run_experiment(
        config_path=str(dst),
        device_str="cuda",
        baseline_logits=str(FP32_LOGITS),
        run_benchmarks=False,
    )
    print(f"\nScheduled QAT ({best}) INT8:")
    for k, v in results_int8.items():
        print(f"  {k:25s}  {v}")
    gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipped (RUN_BEST_INT8=False).")

## 9. Persist all results

In [ ]:
summary_path = Path(WORKING_DIR) / "results" / "scheduled_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary = {
    "int4": {
        "linear": results_linear,
        "cosine": results_cosine,
        "step":   results_step,
    },
    "best_schedule": best,
    "int8_best": results_int8,
    "fp32": fp32,
}
with summary_path.open("w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Wrote {summary_path}")
!ls -lh {WORKING_DIR}/models/checkpoints/

## Next steps

Save outputs as `sqat-scheduled-qat`. Notebook 07 will compare these against PTQ, Standard QAT, and LoRA-QAT.

Proceed to `05_lora_qat.ipynb`.